# MCM LTER Hyporheic Sensor Data Processing

This script was created by Jared Collins in 2025 to be used by the MCM LTER stream team as an easy way to process data from the hyporheic sensors from the stream gages, most of which were installed in the 2425 season. The sensors measure volumetric water content (VWC), conductivity, and temperature in the sediments beneath the stream.

## Importing Packages + Setting Directory

It is very important that the computer being used for processing contains the proper packages, paths, and names for the data files that will be used. The hyporheic data is a little bit more difficult to work with than data processed in Aquarius- unlike data from Aquarius, it is not automatically converted into a dataframe from the .dat file.

**To use this script, please copy the .dat file you would like to convert to your downloads folder.**

Note: The datafiles from the CR1000X all have the same name with no date differentiation, so this can only be done using one data file per stream at a time.

In [9]:
from datetime import date
import pandas as pd
import os
from pathlib import Path
import numpy as np

In [10]:
home = Path.home()
downloads_path = home / "Downloads"
downloads_path = downloads_path.as_posix()
print("Downloads folder:", downloads_path)

Downloads folder: C:/Users/jared/Downloads


In [11]:
streamid_map = {
    'B3': 'lawson_b3',
    'B5': 'bohner_b5',
    'C1': 'common_c1',
    'F1': 'canada_f1',
    'F2': 'huey_f2',
    'F3': 'lostseal_f3',
    'F5': 'aiken_f5',
    'F6': 'vonguerard_f6',
    'F7': 'harnish_f7',
    'F8': 'crescent_f8',
    'F9': 'green_f9',
    'F10': 'delta_f10',
    'H1': 'andersen_h1',
    'M1': 'adams_m1',
    'M2': 'miers_m2',
    'Onyx_LWRT': 'onyx_lwrt',
    'OnyxVanda': 'onyx_vanda'}

## Specifying Data (EDIT THIS SECTION ONLY)

In order for this script to work properly, you will need to input necessary information about the files that you are hoping to publish:
- Stream ID: F1, F2, F3, F5, F6, F7, F8, F9, F10, H1, B3, B5, C1, M1, M2, Onyx_LWRT, OnyxVanda
- Season: Separate seasons into A and B data depending on what portion of the data you are working on; will only impact how datafile is saved. Record the season as the two years followed by the letter (for example, A data from 2022-2023 should be writted as (2223A)
- Date: The first day of the data file, written in "YYYYMMDD" format

**Edit the following cell with the information above. If done correctly, this is the only cell you will have to edit.**

In [90]:
streamid = 'OnyxVanda'
season = '2425'
date = '20241031'

## Importing and Downloading Data

This section will import the datafile into the python script and download a processed version of the file to your downloads folder.

**The next section will not have to be edited so long as the correct information is included above.**

In [91]:
vwc = pd.read_csv(f'{downloads_path}/Soil_Moisture.HZ@{streamid}.{date}.csv', skiprows=14)
cond = pd.read_csv(f'{downloads_path}/Cond.HZ@{streamid}.{date}.csv', skiprows=14)
temp = pd.read_csv(f'{downloads_path}/Soil_Temp.HZ@{streamid}.{date}.csv', skiprows=14)
streamid_lc = streamid.lower()

def clean_sensor_failures(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    bad_values = [7999, -7999, 9999, -9999, '7999', '-7999', '9999', '-9999', '-INF', 'INF','NAN','nan']
    df.replace(bad_values, np.nan, inplace=True)
    df = df.replace([np.inf, -np.inf], np.nan)
    for col in df.select_dtypes(include=[np.number]).columns:
        df[col] = df[col].astype(float)
    return df

def prep_sensor(df, value_name):
    df = clean_sensor_failures(df)
    df['date_time'] = pd.to_datetime(df['Timestamp (UTC+13:00)'])
    df = df[['date_time', 'Value']]
    df = df.rename(columns={'Value': value_name})
    return df
    
vwc = prep_sensor(vwc, 'hzvwc_m3m3')
cond = prep_sensor(cond, 'hzec_dsm')
temp = prep_sensor(temp, 'hztemp_degc')

In [92]:
subm_file = (vwc.merge(cond, on='date_time', how='outer').merge(temp, on='date_time', how='outer'))
full_times = pd.DataFrame({'date_time': pd.date_range(start = subm_file['date_time'].min(), end = subm_file['date_time'].max(),freq = '15min')})
subm_file = full_times.merge(subm_file, on = 'date_time', how = 'left')
subm_file['dataset_code'] = f'strm_hydro_{streamid_lc}'
subm_file['strmgageid'] = streamid_map[streamid]
subm_file = clean_sensor_failures(subm_file)
mask = subm_file['hzvwc_m3m3'] == 0
subm_file.loc[mask, ['hzvwc_m3m3', 'hzec_dsm', 'hztemp_degc']] = np.nan
subm_file = subm_file[['dataset_code', 'strmgageid', 'date_time', 'hzvwc_m3m3', 'hzec_dsm', 'hztemp_degc']]
subm_file.to_csv(f'{downloads_path}/{streamid}_{season}_HZ_SUBM.csv', index = False,na_rep = '')